In [1]:
import numpy as np
import cv2
import joblib
import json
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model


In [2]:
# Loading saved components

svm_model = joblib.load("ecg_svm_model.pkl")
scaler = joblib.load("ecg_scaler.pkl")

with open("class_mapping.json", "r") as f:
    index_to_class = json.load(f)

# Convert string keys to int
index_to_class = {int(k): v for k, v in index_to_class.items()}

print("Model components loaded.")


Model components loaded.


In [3]:
# Rebuild VGG16 feature extractor

from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras import layers

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = layers.Lambda(preprocess_input)(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs, x)


In [4]:
# Cropping Function

def crop_ecg_image(image):
    h, w, _ = image.shape
    top_crop = int(0.18 * h)
    bottom_crop = int(0.95 * h)
    left_crop = int(0.03 * w)
    right_crop = int(0.97 * w)
    return image[top_crop:bottom_crop, left_crop:right_crop]


In [5]:

# Prediction Function
# --------------------------

from tensorflow.keras.applications.vgg16 import preprocess_input

def predict_ecg(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    image = crop_ecg_image(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    image = cv2.resize(image, (224, 224))
    image = np.stack([image, image, image], axis=-1)

    image = image.astype("float32") / 255.0   # IMPORTANT

    image = np.expand_dims(image, axis=0)

    features = feature_extractor.predict(image, verbose=0)
    features_scaled = scaler.transform(features)

    pred_index = svm_model.predict(features_scaled)[0]
    return index_to_class[pred_index]

In [11]:
result = predict_ecg(r"C:\Users\dell\ECG - Cardialyse Final Year Project\ECG_Dataset_Original\Normal Person ECG Images (284x12=3408)\Normal(16).jpg")
print("ECG Diagnosis:", result)


ECG Diagnosis: Normal ECG


In [12]:
result = predict_ecg(r"C:\Users\dell\ECG - Cardialyse Final Year Project\ECG_Dataset_Original\ECG Images of Patient that have abnormal heartbeat (233x12=2796)\HB(100).jpg")
print("ECG Diagnosis:", result)


ECG Diagnosis: Abnormal Heartbeat - Arrhythmias


In [13]:
result = predict_ecg(r"C:\Users\dell\ECG - Cardialyse Final Year Project\ECG_Dataset_Original\ECG Images of Myocardial Infarction Patients (240x12=2880)\MI(75).jpg")
print("ECG Diagnosis:", result)


ECG Diagnosis: Myocardial Infarction


In [9]:
result = predict_ecg(r"C:\Users\dell\ECG - Cardialyse Final Year Project\ECG_Dataset_Original\ECG Images of Patient that have History of MI (172x12=2064)\PMI(19).jpg")
print("ECG Diagnosis:", result)


ECG Diagnosis: History of Myocardial Infarction


In [10]:
result = predict_ecg("arrythmias_internet.jpg")
print("ECG Diagnosis:", result)

ECG Diagnosis: Abnormal Heartbeat - Arrhythmias
